In [ ]:
from dotenv import dotenv_values
from langchain_community.vectorstores.neo4j_vector import Neo4jVector
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from sentence_transformers import CrossEncoder

from Database.embed_models import AsyncSapBERTEmbedder
from Src.run import run_batch, run_batch_top1

c:\ProgramData\Anaconda3\envs\panu\natt_snomed\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
env = dotenv_values(r"H:\NatT\snomed\.env")
neo4j_dict = {
        "uri": env.get('NEO4J_HOST'),
        "username": env.get('NEO4J_USER'),
        "password": env.get('NEO4J_PASSWORD'),
        "gemini_api": env.get('GEMINI_API_KEY'),
        "openai_api" : env.get('OPENAI_API_KEY'),
        "lmstudio_api" : env.get('LMSTUDIO_API'),
        "lmstudio_uri" : env.get('LMSTUDIO_URL'),
        "test_data": env.get('TEST_DATA_PATH')
}


In [4]:
embedding_model = AsyncSapBERTEmbedder(device="cuda")
reranker = CrossEncoder("BAAI/bge-reranker-large")
vector = Neo4jVector.from_existing_index(
    url=neo4j_dict["uri"],
    username=neo4j_dict["username"],
    password=neo4j_dict["password"],
    embedding=embedding_model,
    database="neo4j",
    node_label="Concept",
    text_node_property="text",
    embedding_node_property="embedding",
    index_name="name_embedding"
)
k_candidate = 20
path_testdata = neo4j_dict["test_data"]
pred_repo = "H:\\NatT\\snomed\\DATA\\predictions"

In [ ]:
llm = ChatOpenAI(
            api_key=neo4j_dict["openai_api"],
            model="gpt-4.1-2025-04-14",
            temperature=0,
            max_tokens=None,
            timeout=None,
            max_retries=2,
            )
predict_path = pred_repo + "\\gpt_top1.json"
run_batch_top1(test_path=path_testdata, 
          output_path=predict_path, 
          neo4j_dict=neo4j_dict, 
          k_candidate=k_candidate, 
          llm=llm,
          vector=vector,
          reranker=reranker)
predict_path = pred_repo + "\\gpt.json"
run_batch(test_path=path_testdata, 
          output_path=predict_path, 
          neo4j_dict=neo4j_dict, 
          k_candidate=k_candidate, 
          vector=vector,
          llm=llm, 
          reranker=reranker)

llm = ChatGoogleGenerativeAI(
            api_key=neo4j_dict["gemini_api"],
            model="gemini-2.0-flash",
            temperature=0,
            max_tokens=None,
            timeout=None,
            max_retries=2,
            )
predict_path = pred_repo + "\\gemini.json"
run_batch(test_path=path_testdata, 
          output_path=predict_path, 
          neo4j_dict=neo4j_dict, 
          k_candidate=k_candidate, 
          vector=vector,
          llm=llm, 
          reranker=reranker)

In [ ]:
llm = ChatOpenAI(
            model="medgemma-27b-it",
            base_url="http://localhost:1234/v1/",
            api_key="dummy_key",
            seed=2531
            )
predict_path = pred_repo + "\\medgemma_top1.json"
run_batch_top1(test_path=path_testdata, 
          output_path=predict_path, 
          neo4j_dict=neo4j_dict, 
          k_candidate=k_candidate, 
          llm=llm,
          vector=vector,
          reranker=reranker)
predict_path = pred_repo + "\\medgemma.json"
run_batch(test_path=path_testdata, 
          output_path=predict_path, 
          neo4j_dict=neo4j_dict, 
          k_candidate=k_candidate, 
          vector=vector,
          llm=llm, 
          reranker=reranker)

In [ ]:
llm = ChatOpenAI(
            model="qwen3-30b-a3b-instruct-2507",
            base_url="http://localhost:1234/v1/",
            api_key="dummy_key",
            seed=2531
            )
predict_path = pred_repo + "\\qwen_top1.json"
run_batch_top1(test_path=path_testdata, 
          output_path=predict_path, 
          neo4j_dict=neo4j_dict, 
          k_candidate=k_candidate, 
          llm=llm,
          vector=vector,
          reranker=reranker)
predict_path = pred_repo + "\\qwen.json"
run_batch(test_path=path_testdata, 
          output_path=predict_path, 
          neo4j_dict=neo4j_dict,
          k_candidate=k_candidate,
          vector=vector,
          llm=llm,
          reranker=reranker)